In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
print(model)

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


In [3]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [4]:
v1.shape

(384,)

In [5]:
v1

array([ 2.13903859e-02, -7.39799738e-02,  1.42069475e-03,  2.13816315e-02,
        2.45113391e-02,  3.15582603e-02, -1.10839710e-01, -1.05017453e-01,
       -6.18258938e-02, -6.42312970e-03,  3.72394198e-03,  9.06393528e-02,
       -9.49937198e-03,  6.53976947e-02,  1.10946735e-02, -2.10097395e-02,
       -3.35125700e-02, -4.31677736e-02,  9.96346213e-03,  1.41969677e-02,
       -6.40415177e-02, -7.04179332e-03, -7.91188180e-02,  5.80030978e-02,
        1.30213588e-03,  4.19733720e-03,  5.70979156e-02,  6.39447868e-02,
        2.49903090e-02, -3.95876467e-02, -3.79506387e-02,  2.70394497e-02,
        1.79423485e-02,  1.72272474e-02,  3.43311429e-02,  9.29058157e-03,
        5.86054958e-02, -4.97789867e-02, -5.05369157e-03,  4.34328243e-02,
       -1.56622920e-02, -2.97534615e-02, -5.13324142e-03,  5.13414778e-02,
        6.16064621e-03,  6.86980411e-02, -1.29505135e-02, -5.61938696e-02,
       -1.08265216e-02,  5.96683659e-02,  5.29939905e-02, -3.42754982e-02,
       -4.15274203e-02, -

In [6]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [7]:
v1.dot(dv)

np.float32(0.32332397)

In [8]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [9]:
v2.dot(dv)

np.float32(0.019730574)

In [11]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-29 20:08:24--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 888 [text/plain]
Saving to: ‘ingest.py’

ingest.py           100%[===================>]     888  --.-KB/s    in 0s      

2026-06-29 20:08:24 (56.2 MB/s) - ‘ingest.py’ saved [888/888]



In [12]:
from ingest import load_faq_data

documents = load_faq_data()

In [13]:
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

In [14]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [15]:
len(texts)

1350

In [16]:
from tqdm.auto import tqdm

In [17]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [18]:
import numpy as np
X = np.array(vectors)

In [19]:
X.shape

(1350, 384)